# Phase 2: build the training data (no GPU needed) - v1.2

Builds about 30,000 physics examples with **complete, short thinking traces** for LFM2.5-1.2B-Thinking.

**v1.1** made the subjects follow the exam (quantum, atomic and statistical physics are 45% of UGPhysics) and added NaturalReasoning physics.

**New in v1.2** (after reading the v1.1 outputs)
- Stricter leak check: a reworded copy of an exam problem that keeps the same numbers is now removed even below the TF-IDF threshold (v1.1 kept a UGPhysics relativity problem at similarity 0.40).
- Smarter answer check: `N_{\text{est}}` = `N_{est}`, `E = 5` = `5`, and "(a) ... (b) ..." references are split into parts. v1.1 rejected 39% of checked items, many only because of notation, and that hit quantum and statistical physics hardest.
- True/False answers are boxed as True/False (what the UGPhysics prompt asks), PhysicsEval problems that need a figure are dropped, and stats.md shows where each subject loses examples.

**Before you start**
1. Replace `physdata.py` in the Drive folder with version 1.2. Keep `physeval.py` there.
2. **Runtime > Change runtime type > CPU.** No GPU quota is used.
3. Run the cells in order. Nothing is downloaded again (the caches from v1.1 are reused), and earlier answer checks are reused too, so it takes about 20 to 40 minutes.

## 1. Settings

In [ ]:
WORKDIR = "/content/drive/MyDrive/physics_eval"   # same folder as the evaluation
NAME = "v1.2"                                     # output goes to WORKDIR/train_data/v1.2
N_TOTAL = 30000                                   # training + validation examples

## 2. Mount Drive and check files

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
for f in ["physeval.py", "physdata.py", "data/items_ugphysics.jsonl"]:
    assert os.path.exists(f"{WORKDIR}/{f}"), f"missing {f} in {WORKDIR}"
print("ok")

## 3. Install packages

In [ ]:
!pip -q install -U datasets transformers huggingface_hub scipy "antlr4-python3-runtime==4.11.1" chardet

## 4. Check the grader (must end with SELF-TEST PASSED)

In [ ]:
!python "{WORKDIR}/physeval.py" selftest --workdir "{WORKDIR}"

## 5. Build the data
It prints 8 steps: exam questions to protect, sources (the one-time second download), **topic classifiers** (with their cross-validation scores), building examples, **removing anything that resembles an exam question**, token lengths, **subject quotas + the official grader check**, writing.

If Colab disconnects during the download, just run this cell again. Nothing is cached until the download finishes.

In [ ]:
!python "{WORKDIR}/physdata.py" build --workdir "{WORKDIR}" --name {NAME} --n-total {N_TOTAL}

## 6. Read the summary
Check these sections:
- **Subjects vs the exam**: the "share of physics items" column should now be close to the "target" column (quantum, atomic and statistical physics should no longer be tiny).
- **Classifiers**: the physics filter should accept almost all physics and very little of the other subjects. The subject tagger table shows which tagger was used.
- **Where each subject's examples are lost**: shows whether a thin subject is short of candidates, loses them to the length limit, or to the answer check.
- **Grader check** and **Lengths**: thinking should stay far below the base model's 4,096 tokens.

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f"{WORKDIR}/train_data/{NAME}/stats.md").read()))

## 7. Read real examples
Skim these. The reasoning should be complete and the final answer correct. If something looks wrong, send me a few examples before training.

In [ ]:
display(Markdown(open(f"{WORKDIR}/train_data/{NAME}/samples.md").read()))

## 8. Look at the closest leak decisions
Rows marked `kept (below threshold)` were allowed in. If any of them is clearly the same problem as the exam question, tell me and I'll lower the threshold.

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 160)
leaks = pd.read_csv(f"{WORKDIR}/train_data/{NAME}/leak_report.csv")
print(leaks["decision"].str.split(" ").str[0].value_counts())
leaks[leaks["decision"].str.startswith("kept")].head(15)

## 9. Look at answers the grader rejected
These examples were removed because the worked solution's own final answer disagrees with the reference answer. If many rows look like the **same** answer written differently, send me a few; that means the check is too strict.

In [ ]:
drops = pd.read_csv(f"{WORKDIR}/train_data/{NAME}/grader_drops.csv")
print(len(drops), "rejected;", drops["pool"].value_counts().to_dict())
drops.sample(min(15, len(drops)), random_state=0)[["reference_answer", "solution_answer", "question"]]